# Flight Departure Delay Predictor — Enhanced with Lightning Warnings

**Extension of the baseline SATS departure-delay model that incorporates lightning warning (LW) data.**

### What's new vs the baseline notebook
- **Lightning Warning feature engineering** — daily LW count/intensity, overlap between LW windows and the ground-handling window, active-LW flag at scheduled departure time
- **Side-by-side comparison** — baseline XGBoost vs LW-enhanced XGBoost
- **Feature importance** — quantifies how much the new LW features contribute

### Data sources
1. `dsmlive_milestone_Nov2025.xlsx` + `dsmlive_flight_Nov2025.xlsx` (old batch)
2. `dsmlive_GHAMS_01022026_07042026.csv` + `dsmlive_flight_01022026_07042026.csv` (new batch)
3. `lw_2024_2025.csv` — Lightning warning windows Jan 2024 – Dec 2025


## 1. Imports & Data Paths

In [ ]:
import pandas as pd
import numpy as np
import ast, json, re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import pickle, warnings
warnings.filterwarnings('ignore')

BASE = '/Users/prathosh/Desktop/Sats Project/Data'
path_old_milestone = f'{BASE}/Old DSM Data/dsmlive_milestone_Nov2025.xlsx'
path_old_flight    = f'{BASE}/Old DSM Data/dsmlive_flight_Nov2025.xlsx'
path_new_milestone = f'{BASE}/New DSM Data/dsmlive_GHAMS_01022026_07042026.csv'
path_new_flight    = f'{BASE}/New DSM Data/dsmlive_flight_01022026_07042026.csv'
path_lw            = f'{BASE}/lw_2024_2025.csv'
path_model_out     = '/Users/prathosh/Desktop/Sats Project/data/model_with_lw.pkl'

print('Libraries loaded successfully.')


## 2. Lightning Warning (LW) Data

### Data dictionary
| Column | Description |
|--------|-------------|
| `LWNo` | Warning number for that day (sequential 1–64). Codes ≥ 10 000 (50000, 21000, 17000, 15000, 13000) are **midnight-continuation** rows — the window from the previous day crosses midnight, so the continuation row's `DateIssued` is the *next* day and `LW_ValidFrom = 00:00:00`. |
| `LW_ValidFrom` | Warning window start — `HH:MM:SS` |
| `LW_ValidTo` | Warning window end — `HH:MM:SS` (usually 30 min after start) |
| `Status` | `NaN` = active; `Cancelled` = cancelled |
| `DateIssued` | Date the warning was issued — `DD/MM/YYYY` |
| `Issued` | Minutes:seconds offset (redundant / corrupted) — **ignore** |
| `TimeIssued` | Exact time warning was issued — `HH:MM:SS` (use this) |


In [ ]:
# --- Load ---
lw_raw = pd.read_csv(path_lw)
lw_raw.columns = lw_raw.columns.str.strip()   # strip leading/trailing spaces
print(f'Raw shape: {lw_raw.shape}')
print(f'Columns  : {list(lw_raw.columns)}')
lw_raw.head(10)


In [ ]:
# --- Quick inspection ---
print('=== Data types ===')
print(lw_raw.dtypes)
print()
print('=== Status value counts ===')
print(lw_raw['Status'].value_counts(dropna=False))
print()
print('=== LWNo special codes (midnight continuations) ===')
print(lw_raw[lw_raw['LWNo'] >= 10000][['LWNo','DateIssued','LW_ValidFrom','LW_ValidTo']].drop_duplicates())


### 2a. Parse, deduplicate, and clean

In [ ]:
def parse_lw_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full parse + clean pipeline for the lightning warning CSV.
    Returns one row per unique active warning window.
    """
    df = df.copy()

    # 1. Drop Cancelled warnings
    df = df[df['Status'].isna()].copy()
    print(f'After dropping Cancelled: {len(df):,} rows')

    # 2. Deduplicate — same warning sent to multiple recipients
    df = df.drop_duplicates(subset=['LWNo', 'DateIssued', 'LW_ValidFrom', 'LW_ValidTo'])
    print(f'After deduplication     : {len(df):,} rows')

    # 3. Parse DateIssued (DD/MM/YYYY)
    df['date'] = pd.to_datetime(df['DateIssued'], dayfirst=True).dt.date

    # 4. Parse time strings HH:MM:SS -> timedelta from midnight
    def to_td(s):
        """'HH:MM:SS' or '0:MM:SS' -> pd.Timedelta"""
        try:
            parts = str(s).strip().split(':')
            h, m, sec = int(parts[0]), int(parts[1]), int(float(parts[2]))
            return pd.Timedelta(hours=h, minutes=m, seconds=sec)
        except Exception:
            return pd.NaT

    df['td_from'] = df['LW_ValidFrom'].apply(to_td)
    df['td_to']   = df['LW_ValidTo'].apply(to_td)

    # 5. Build full datetimes
    base = pd.to_datetime(df['date'].astype(str))   # midnight of DateIssued
    df['lw_start'] = base + df['td_from']
    df['lw_end']   = base + df['td_to']

    # 6. Handle midnight-crossing windows
    #    Continuation rows (LWNo >= 10000): DateIssued IS the next day;
    #    lw_start = midnight of that day, lw_end = the ValidTo time on that day.
    #    The original window on the PREVIOUS day already ends at 23:59:59 —
    #    the continuation row extends the coverage from 00:00 onward.
    #    Both rows are kept as separate (non-overlapping) intervals.
    is_continuation = df['LWNo'] >= 10000
    print(f'Midnight-continuation rows: {is_continuation.sum()}')

    # 7. Drop rows where end <= start (malformed)
    df = df[df['lw_end'] > df['lw_start']]

    # 8. Duration
    df['lw_duration_mins'] = (df['lw_end'] - df['lw_start']).dt.total_seconds() / 60

    keep_cols = ['LWNo', 'date', 'lw_start', 'lw_end', 'lw_duration_mins']
    df = df[keep_cols].reset_index(drop=True)
    print(f'Final clean LW rows     : {len(df):,}')
    return df

lw = parse_lw_data(lw_raw)
print()
lw.head(10)


### 2b. EDA — Lightning Warning patterns

In [ ]:
# --- Monthly LW frequency ---
lw['year_month'] = pd.to_datetime(lw['date']).dt.to_period('M')
monthly_counts = lw.groupby('year_month').size().reset_index(name='lw_count')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Monthly count
axes[0].bar(monthly_counts['year_month'].astype(str), monthly_counts['lw_count'], color='steelblue')
axes[0].set_title('Monthly Active Lightning Warning Count')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Number of LW Windows')
axes[0].tick_params(axis='x', rotation=45)

# Duration distribution
axes[1].hist(lw['lw_duration_mins'], bins=20, color='tomato', edgecolor='white')
axes[1].set_title('LW Window Duration Distribution')
axes[1].set_xlabel('Duration (minutes)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(30, color='black', linestyle='--', label='30-min mark')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Duration stats:\n{lw["lw_duration_mins"].describe().round(1)}')


In [ ]:
# --- Hour-of-day distribution of LW starts ---
lw['lw_start_hour'] = pd.to_datetime(lw['lw_start']).dt.hour
hour_counts = lw.groupby('lw_start_hour').size()

plt.figure(figsize=(10, 3))
plt.bar(hour_counts.index, hour_counts.values, color='darkorange', edgecolor='white')
plt.title('Lightning Warnings by Hour of Day')
plt.xlabel('Hour (local — adjust UTC offset if needed)')
plt.ylabel('Number of LW Windows')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()


### 2c. Lightning Warning feature engineering

We create **two tiers** of features:

**Tier 1 — Daily aggregates** (joined by departure date, no flight-level times needed):
- `LW_Count_On_Date` — number of active LW windows on the departure date
- `Total_LW_Mins_On_Date` — total LW coverage in minutes
- `LW_Day_Had_Warning` — binary flag (any LW that day?)

**Tier 2 — Flight-level features** (require ground-handling window times):
- `LW_Active_At_Sched_Departure` — was any LW active at the exact scheduled departure time?
- `LW_Overlap_With_Ground_Window_Mins` — total overlap between all LW windows and the ground-handling window `[arrival.actual → departure.scheduled]`
- `LW_Active_During_Ground_Time` — binary flag (any LW overlap > 0?)
- `Mins_Since_Last_LW_Before_Dep` — minutes since the most-recent LW ended before scheduled departure (0 if LW still active; NaN if no LW that day)


In [ ]:
# --- Tier 1: daily aggregates ---
lw_daily = (
    lw.groupby('date')
    .agg(
        LW_Count_On_Date       = ('LWNo', 'count'),
        Total_LW_Mins_On_Date  = ('lw_duration_mins', 'sum'),
    )
    .reset_index()
)
lw_daily['LW_Day_Had_Warning'] = 1

print(f'Daily LW summary: {len(lw_daily)} unique days with active LWs')
print(lw_daily.describe().round(1))


In [ ]:
def compute_flight_lw_features(flight_date, dep_scheduled, arr_actual, lw_df):
    """
    Compute Tier-2 LW features for a single flight.
    
    Parameters
    ----------
    flight_date     : date object — departure date (used to filter lw_df)
    dep_scheduled   : Timestamp — scheduled off-block
    arr_actual      : Timestamp — actual inbound in-block
    lw_df           : cleaned LW DataFrame with lw_start, lw_end columns

    Returns
    -------
    dict of feature values
    """
    # Default values (no LW data for this flight)
    result = {
        'LW_Active_At_Sched_Departure': 0,
        'LW_Overlap_With_Ground_Window_Mins': 0.0,
        'LW_Active_During_Ground_Time': 0,
        'Mins_Since_Last_LW_Before_Dep': np.nan,
    }

    # Filter LWs for this date
    day_lws = lw_df[lw_df['date'] == flight_date]
    if day_lws.empty:
        return result

    # Ensure datetimes are timezone-naive for comparison
    def strip_tz(ts):
        if pd.isna(ts): return pd.NaT
        if hasattr(ts, 'tzinfo') and ts.tzinfo is not None:
            return ts.tz_convert(None)
        return ts

    dep_sched = strip_tz(dep_scheduled)
    arr_act   = strip_tz(arr_actual)

    # --- LW active at scheduled departure ---
    if pd.notna(dep_sched):
        active_at_dep = day_lws[
            (day_lws['lw_start'] <= dep_sched) & (day_lws['lw_end'] >= dep_sched)
        ]
        result['LW_Active_At_Sched_Departure'] = int(len(active_at_dep) > 0)

    # --- Overlap with ground-handling window ---
    if pd.notna(arr_act) and pd.notna(dep_sched) and arr_act < dep_sched:
        gw_start, gw_end = arr_act, dep_sched
        total_overlap = 0.0
        for _, lw_row in day_lws.iterrows():
            # Overlap = max(0, min(ends) - max(starts))
            overlap_start = max(gw_start, lw_row['lw_start'])
            overlap_end   = min(gw_end,   lw_row['lw_end'])
            if overlap_end > overlap_start:
                total_overlap += (overlap_end - overlap_start).total_seconds() / 60
        result['LW_Overlap_With_Ground_Window_Mins'] = round(total_overlap, 2)
        result['LW_Active_During_Ground_Time']       = int(total_overlap > 0)

    # --- Minutes since last LW ended before scheduled departure ---
    if pd.notna(dep_sched):
        ended_before = day_lws[day_lws['lw_end'] <= dep_sched]
        if not ended_before.empty:
            last_end = ended_before['lw_end'].max()
            result['Mins_Since_Last_LW_Before_Dep'] = (
                (dep_sched - last_end).total_seconds() / 60
            )

    return result

print('Flight-level LW feature function defined.')


## 3. Flight Data — Load & Parse

In [ ]:
# Load raw files
old_milestone = pd.read_excel(path_old_milestone)
old_flight    = pd.read_excel(path_old_flight)
new_milestone = pd.read_csv(path_new_milestone)
new_flight    = pd.read_csv(path_new_flight, engine='python', on_bad_lines='skip')

print(f'Old milestone : {old_milestone.shape}')
print(f'Old flight    : {old_flight.shape}')
print(f'New milestone : {new_milestone.shape}')
print(f'New flight    : {new_flight.shape}')


In [ ]:
# --- Shared dict parser ---
def parse_dict(val):
    if pd.isna(val): return {}
    if isinstance(val, str):
        try: return json.loads(val)
        except (ValueError, TypeError):
            try: return ast.literal_eval(val)
            except (ValueError, SyntaxError): return {}
    return val

# --- Flatten OLD milestone ---
old_milestone['milestones'] = old_milestone['milestones'].apply(parse_dict)
milestones_expanded = pd.json_normalize(old_milestone['milestones']).add_prefix('milestone_')
old_milestone_clean = pd.concat([old_milestone.drop(columns=['milestones']), milestones_expanded], axis=1)
print('Old milestone flattened:', old_milestone_clean.shape)

# --- Flatten NEW milestone ---
nested_milestone_cols = ['milestones', 'specialHandling']
new_milestone_clean   = new_milestone.copy()
for col in nested_milestone_cols:
    if col not in new_milestone_clean.columns: continue
    new_milestone_clean[col] = new_milestone_clean[col].apply(parse_dict)
    first = new_milestone_clean[col].dropna().iloc[0] if not new_milestone_clean[col].dropna().empty else None
    if isinstance(first, dict):
        exp = pd.json_normalize(new_milestone_clean[col]).add_prefix(f'{col}_')
        new_milestone_clean = pd.concat([new_milestone_clean.drop(columns=[col]), exp], axis=1)
print('New milestone flattened:', new_milestone_clean.shape)


In [ ]:
# --- Flatten flight datasets ---
nested_flight_cols = ['identification','transitPoints','status','aircraft','arrival','departure','origin','destination','linkedFlight']

def flatten_flight(df):
    df = df.copy()
    for col in nested_flight_cols:
        if col not in df.columns: continue
        df[col] = df[col].apply(parse_dict)
        first = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
        if isinstance(first, dict):
            exp = pd.json_normalize(df[col]).add_prefix(f'{col}_')
            df = pd.concat([df.drop(columns=[col]), exp], axis=1)
    if 'changes' in df.columns:
        df['changes'] = df['changes'].apply(parse_dict)
    return df

old_flight_clean = flatten_flight(old_flight)
new_flight_clean = flatten_flight(new_flight)
print('Old flight flattened:', old_flight_clean.shape)
print('New flight flattened:', new_flight_clean.shape)


In [ ]:
# --- Merge flight + milestone ---
old_merged = pd.merge(old_flight_clean, old_milestone_clean, on='id', how='inner', suffixes=('_flight','_milestone'))
new_merged = pd.merge(new_flight_clean, new_milestone_clean, on='id', how='inner', suffixes=('_flight','_milestone'))
print(f'Old merged: {old_merged.shape}')
print(f'New merged: {new_merged.shape}')

# Fix prefix inconsistency in new data
new_merged.columns = [c.replace('milestones_','milestone_') if c.startswith('milestones_') else c for c in new_merged.columns]

# Drop database metadata columns
for _df in [old_merged, new_merged]:
    meta = [c for c in _df.columns if c.startswith('_')]
    _df.drop(columns=meta, inplace=True, errors='ignore')

# Stack
final_combined_data = pd.concat([old_merged, new_merged], ignore_index=True)
print(f'Combined shape: {final_combined_data.shape}')


In [ ]:
# --- Convert ISO-8601 datetime strings ---
df = final_combined_data.copy()
converted = 0
for col in df.select_dtypes(include=['object']).columns:
    first = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
    if isinstance(first, str) and 'T' in first and first.endswith('Z'):
        df[col] = pd.to_datetime(df[col], errors='coerce')
        converted += 1
print(f'Converted {converted} datetime columns.')


## 4. Feature Engineering

In [ ]:
# --- Filter to Departures only ---
df_dep = df[df['identification_direction'] == 'Departure'].copy()
print(f'Departure rows: {len(df_dep):,}')

# Ensure key datetime columns are parsed
for col in ['linkedFlight_arrival.inBlock.actual','departure_offBlock.scheduled',
            'departure_offBlock.actual','linkedFlight_arrival.inBlock.scheduled']:
    if col in df_dep.columns:
        df_dep[col] = pd.to_datetime(df_dep[col], errors='coerce')


In [ ]:
# --- Helper: clean PTS strings like '-5|-5' -> float ---
def clean_pts_column(series):
    s = series.astype(str).str.split('|').str[0].str.replace('+','', regex=False)
    s = s.replace(['None','nan','NaN','null','Null',''], np.nan)
    return s.astype(float)

def calculate_planned_time(df_temp, base_col_name):
    actual_col = f'{base_col_name}.actual'
    pts_col    = f'{base_col_name}.PTS'
    ori_col    = f'{base_col_name}.orientation'
    if not all(c in df_temp.columns for c in [actual_col, pts_col, ori_col]):
        return pd.Series(pd.NaT, index=df_temp.index)
    pts_minutes    = clean_pts_column(df_temp[pts_col])
    is_arr_ref     = df_temp[ori_col].isin(['A/T','A','T'])
    is_dep_ref     = df_temp[ori_col].isin(['D','T/D'])
    ref_time       = pd.Series(pd.NaT, index=df_temp.index)
    ref_time       = np.where(is_arr_ref, df_temp['linkedFlight_arrival.inBlock.actual'],   ref_time)
    ref_time       = np.where(is_dep_ref, df_temp['departure_offBlock.scheduled'],          ref_time)
    return pd.to_datetime(ref_time) + pd.to_timedelta(pts_minutes, unit='m')

# --- Identify and compute analysis columns ---
all_actuals  = [c for c in df_dep.columns if c.startswith('milestone_') and c.endswith('.actual')]
start_cols   = [c for c in all_actuals if '_start' in c]
end_cols     = [c for c in all_actuals if '_end'   in c]
point_cols   = [c for c in all_actuals if c not in start_cols and c not in end_cols]

activities, used_ends = [], set()
for sc in start_cols:
    parts    = sc.split('.')
    category = parts[0]
    s_event  = parts[1]
    core     = s_event.replace('_start','').replace('first','').lower()
    matches  = [e for e in end_cols if e.startswith(category) and core in e.lower() and e not in used_ends]
    if matches:
        ec = matches[0]; used_ends.add(ec)
        activities.append((sc, ec))

for (sc, ec) in activities:
    base  = sc.replace('.actual','')
    ebase = ec.replace('.actual','')
    df_dep[sc] = pd.to_datetime(df_dep[sc], errors='coerce')
    df_dep[ec] = pd.to_datetime(df_dep[ec], errors='coerce')
    act_dur_col  = f'{base}_analysis_ActualDuration_mins'
    plan_dur_col = f'{base}_analysis_PlannedDuration_mins'
    delay_col    = f'{base}_analysis_Delay_mins'
    df_dep[act_dur_col]  = (df_dep[ec] - df_dep[sc]).dt.total_seconds() / 60
    planned_start = calculate_planned_time(df_dep, base)
    planned_end   = calculate_planned_time(df_dep, ebase)
    df_dep[plan_dur_col] = (planned_end - planned_start).dt.total_seconds() / 60
    df_dep[delay_col]    = df_dep[act_dur_col] - df_dep[plan_dur_col]

for pc in point_cols:
    base      = pc.replace('.actual','')
    delay_col = f'{base}_analysis_Delay_mins'
    df_dep[pc]        = pd.to_datetime(df_dep[pc], errors='coerce')
    planned_time      = calculate_planned_time(df_dep, base)
    df_dep[delay_col] = (df_dep[pc] - planned_time).dt.total_seconds() / 60

print('Milestone analysis columns computed.')


In [ ]:
# --- Target variable + time/operational features ---
df_dep['Target_Departure_Delay_mins'] = (
    df_dep['departure_offBlock.actual'] - df_dep['departure_offBlock.scheduled']
).dt.total_seconds() / 60

bins   = [-np.inf, 0, 4, np.inf]
labels = ['On-Time', 'Acceptable', 'Delayed']
df_dep['Target_Departure_Delay_Class'] = pd.cut(
    df_dep['Target_Departure_Delay_mins'], bins=bins, labels=labels
)

df_dep['Incoming_Delay_mins']      = (
    df_dep['linkedFlight_arrival.inBlock.actual'] - df_dep['linkedFlight_arrival.inBlock.scheduled']
).dt.total_seconds() / 60

df_dep['Hour_of_Day']              = df_dep['departure_offBlock.scheduled'].dt.hour
df_dep['Month']                    = df_dep['departure_offBlock.scheduled'].dt.month
df_dep['Day_of_Week']              = df_dep['departure_offBlock.scheduled'].dt.day_name()
df_dep['Is_Weekend']               = df_dep['Day_of_Week'].isin(['Saturday','Sunday']).astype(int)
df_dep['Available_Ground_Time_mins'] = (
    df_dep['departure_offBlock.scheduled'] - df_dep['linkedFlight_arrival.inBlock.actual']
).dt.total_seconds() / 60
df_dep['Is_Ground_Time_Deficient'] = (
    df_dep['Available_Ground_Time_mins'] < df_dep['status_minGroundTime']
).astype(int)

print('Target and operational features engineered.')
print(df_dep['Target_Departure_Delay_Class'].value_counts(dropna=False))


## 5. Merge Lightning Warning Features onto Flights

> **Note:** `departure_offBlock.scheduled` in the flight data may be UTC. The LW timestamps are local Singapore time (SGT = UTC+8). If timestamps are stored in UTC, convert them before merging — set `UTC_OFFSET_HOURS` accordingly.


In [ ]:
# ── Timezone alignment ──────────────────────────────────────────────────────
# The LW data has NO timezone info — times are in local SGT (UTC+8).
# Check whether flight timestamps are UTC or timezone-naive.
sample_ts = df_dep['departure_offBlock.scheduled'].dropna().iloc[0]
print(f'Sample departure timestamp: {sample_ts}  (tzinfo={getattr(sample_ts,"tzinfo",None)})')

# If timestamps are UTC (tzinfo not None), convert to SGT for matching.
# If already timezone-naive, assume they are already in SGT.
UTC_OFFSET_HOURS = 8   # SGT = UTC+8; set to 0 if data is already in local time

def to_local(ts):
    if pd.isna(ts): return pd.NaT
    if hasattr(ts, 'tzinfo') and ts.tzinfo is not None:
        # Convert UTC -> local naive
        return ts.tz_convert(None) + pd.Timedelta(hours=UTC_OFFSET_HOURS)
    return ts

df_dep['dep_sched_local'] = df_dep['departure_offBlock.scheduled'].apply(to_local)
df_dep['arr_actual_local'] = df_dep['linkedFlight_arrival.inBlock.actual'].apply(to_local)
df_dep['dep_date_local']  = df_dep['dep_sched_local'].dt.date

print(f'Local departure date range: {df_dep["dep_date_local"].min()} to {df_dep["dep_date_local"].max()}')
print(f'LW date range             : {lw["date"].min()} to {lw["date"].max()}')


In [ ]:
# ── Tier 1: daily aggregates join (fast) ────────────────────────────────────
df_dep = df_dep.merge(
    lw_daily.rename(columns={'date':'dep_date_local'}),
    on='dep_date_local', how='left'
)
df_dep['LW_Count_On_Date']      = df_dep['LW_Count_On_Date'].fillna(0).astype(int)
df_dep['Total_LW_Mins_On_Date'] = df_dep['Total_LW_Mins_On_Date'].fillna(0.0)
df_dep['LW_Day_Had_Warning']    = df_dep['LW_Day_Had_Warning'].fillna(0).astype(int)

print(f'Flights on days WITH active LW  : {(df_dep["LW_Day_Had_Warning"]==1).sum():,}')
print(f'Flights on days without LW      : {(df_dep["LW_Day_Had_Warning"]==0).sum():,}')


In [ ]:
# ── Tier 2: flight-level LW features (iterative, may take ~1-2 min) ─────────
print('Computing flight-level LW features...')

# Build a date-indexed dict for fast lookup
lw_by_date = {d: grp for d, grp in lw.groupby('date')}

tier2_records = []
for _, row in df_dep[['dep_date_local','dep_sched_local','arr_actual_local']].iterrows():
    day_lw = lw_by_date.get(row['dep_date_local'], pd.DataFrame())
    rec = compute_flight_lw_features(
        flight_date   = row['dep_date_local'],
        dep_scheduled = row['dep_sched_local'],
        arr_actual    = row['arr_actual_local'],
        lw_df         = day_lw if not day_lw.empty else pd.DataFrame(columns=lw.columns)
    )
    tier2_records.append(rec)

tier2_df = pd.DataFrame(tier2_records, index=df_dep.index)
df_dep   = pd.concat([df_dep, tier2_df], axis=1)

print('Done!')
print('LW feature summary:')
lw_feat_cols = ['LW_Count_On_Date','Total_LW_Mins_On_Date','LW_Day_Had_Warning',
                'LW_Active_At_Sched_Departure','LW_Overlap_With_Ground_Window_Mins',
                'LW_Active_During_Ground_Time','Mins_Since_Last_LW_Before_Dep']
print(df_dep[lw_feat_cols].describe().round(2))


In [ ]:
# --- EDA: LW impact on delay rates ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

valid = df_dep.dropna(subset=['Target_Departure_Delay_Class','LW_Day_Had_Warning'])

# Delay class distribution: LW day vs non-LW day
for ax, flag, title in zip(
    [axes[0], axes[1]],
    [0, 1],
    ['No Lightning Warning Day', 'Lightning Warning Day']
):
    grp = valid[valid['LW_Day_Had_Warning'] == flag]
    cts = grp['Target_Departure_Delay_Class'].value_counts().reindex(['On-Time','Acceptable','Delayed'], fill_value=0)
    ax.bar(cts.index, cts.values / cts.sum() * 100, color=['#2ecc71','#f39c12','#e74c3c'], edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel('Share (%)')
    ax.set_ylim(0, 100)

# Avg departure delay: LW overlap bins
valid2 = df_dep.dropna(subset=['Target_Departure_Delay_mins','LW_Overlap_With_Ground_Window_Mins'])
valid2['LW_Overlap_Bin'] = pd.cut(valid2['LW_Overlap_With_Ground_Window_Mins'],
                                   bins=[-0.01, 0, 10, 20, 999],
                                   labels=['0 (no overlap)', '1-10 min', '11-20 min', '>20 min'])
overlap_avg = valid2.groupby('LW_Overlap_Bin')['Target_Departure_Delay_mins'].mean()
axes[2].bar(overlap_avg.index.astype(str), overlap_avg.values, color='steelblue', edgecolor='white')
axes[2].set_title('Avg Departure Delay by LW Overlap')
axes[2].set_ylabel('Avg Delay (min)')
axes[2].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()


## 6. Assemble Final Prediction DataFrame

In [ ]:
engineered_cols = [
    'Target_Departure_Delay_mins', 'Target_Departure_Delay_Class',
    'Incoming_Delay_mins', 'Hour_of_Day', 'Month', 'Day_of_Week',
    'Is_Weekend', 'Available_Ground_Time_mins', 'Is_Ground_Time_Deficient',
    # Lightning warning features
    'LW_Count_On_Date', 'Total_LW_Mins_On_Date', 'LW_Day_Had_Warning',
    'LW_Active_At_Sched_Departure', 'LW_Overlap_With_Ground_Window_Mins',
    'LW_Active_During_Ground_Time', 'Mins_Since_Last_LW_Before_Dep',
]

base_cols = [
    'id', 'identification_carrierCode', 'identification_iata', 'identification_direction',
    'status_isRemoteBay', 'status_minGroundTime', 'status_actualGroundTime', 'status_delayTime',
    'aircraft_bodyType', 'aircraft_nature', 'aircraft_typeICAO',
    'departure_offBlock.estimated', 'departure_offBlock.scheduled', 'departure_offBlock.actual',
    'departure_takeOff.actual', 'departure_offBlock.target', 'origin_terminal',
    'destination_iata', 'linkedFlight_origin.iata', 'linkedFlight_arrival.inBlock.actual',
    'linkedFlight_arrival.inBlock.estimated', 'linkedFlight_arrival.inBlock.scheduled',
    'linkedFlight_arrival.landing.actual', 'linkedFlight_arrival.landing.estimated',
    'carrierCode', 'direction'
]
base_cols_present = [c for c in base_cols if c in df_dep.columns]

milestone_raw_cols  = [c for c in df_dep.columns if c.startswith('milestone_') and any(t in c for t in ['.actual','.reference','.PTS','.orientation'])]
new_analysis_cols   = [c for c in df_dep.columns if '_analysis_' in c]
special_handling_cols = [c for c in df_dep.columns if c.startswith('specialHandling_')]

all_model_cols = list(set(base_cols_present + milestone_raw_cols + new_analysis_cols + special_handling_cols + engineered_cols))
df_predict     = df_dep[[c for c in all_model_cols if c in df_dep.columns]].copy()

print(f'Prediction DataFrame: {df_predict.shape}')
print('Class distribution:')
print(df_predict['Target_Departure_Delay_Class'].value_counts(dropna=False))


## 7. Model Training

We train **two identical XGBoost + SMOTE pipelines**:
1. **Baseline** — no lightning warning features (identical to the original notebook)
2. **LW-Enhanced** — with the 7 new LW features

Comparing them directly shows how much LW data improves predictive performance.


In [ ]:
def build_feature_matrix(df_input, exclude_lw=False):
    """
    Prepare X, y for model training.
    Set exclude_lw=True to build the baseline (no LW features).
    """
    df = df_input.copy()

    # New interaction features
    df['Ground_Time_Ratio'] = (
        df['Available_Ground_Time_mins'] / df['status_minGroundTime'].replace(0, np.nan)
    ).replace([np.inf, -np.inf], 999).fillna(-999)
    df['Delay_Pressure_Score'] = (
        df['Incoming_Delay_mins'].fillna(0) * df['Is_Ground_Time_Deficient'].fillna(0)
    )

    df = df.dropna(subset=['Target_Departure_Delay_Class'])

    # Anti-leakage purge
    df = df.drop(columns=['Target_Departure_Delay_mins','status_delayTime','status_actualGroundTime'], errors='ignore')
    time_leak = [c for c in df.columns if any(t in c for t in ['.actual','.scheduled','.estimated','.target','.predicted','.PTS','.reference'])]
    df = df.drop(columns=time_leak, errors='ignore')
    df = df.drop(columns=['id','lid','dep_sched_local','arr_actual_local','dep_date_local'], errors='ignore')
    df = df.drop(columns=[c for c in df.columns if '_Text' in c], errors='ignore')
    leakage_kw = ['thumbsUp','cabinDoor_close','PLB_retract','paxStep_retract','cargoDoor_close','finalReadback','NOTOC','loadsheetACK']
    terminal_cols = [c for c in df.columns if any(kw in c for kw in leakage_kw)]
    df = df.drop(columns=terminal_cols, errors='ignore')

    if exclude_lw:
        lw_features = ['LW_Count_On_Date','Total_LW_Mins_On_Date','LW_Day_Had_Warning',
                       'LW_Active_At_Sched_Departure','LW_Overlap_With_Ground_Window_Mins',
                       'LW_Active_During_Ground_Time','Mins_Since_Last_LW_Before_Dep']
        df = df.drop(columns=[c for c in lw_features if c in df.columns], errors='ignore')

    X = df.drop(columns=['Target_Departure_Delay_Class'])
    le_ = LabelEncoder()
    y   = le_.fit_transform(df['Target_Departure_Delay_Class'])

    num_feats = X.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_feats = X.select_dtypes(include=['object','category']).columns.tolist()
    X[cat_feats] = X[cat_feats].astype(str)

    print(f'Features: {len(X.columns)} total ({len(num_feats)} numeric, {len(cat_feats)} categorical)')
    print(f'Class dist: {dict(zip(le_.classes_, np.bincount(y)))}')
    return X, y, le_, num_feats, cat_feats

print('Feature builder ready.')


In [ ]:
def train_pipeline(X, y, num_feats, cat_feats, label='Model'):
    num_t = ImbPipeline([('imputer', SimpleImputer(strategy='constant', fill_value=-999)),
                         ('scaler',  StandardScaler())])
    cat_t = ImbPipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
                         ('onehot',  OneHotEncoder(handle_unknown='ignore'))])
    preprocessor = ColumnTransformer([('num', num_t, num_feats), ('cat', cat_t, cat_feats)])

    pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', xgb.XGBClassifier(
            objective='multi:softprob', num_class=3, eval_metric='mlogloss',
            missing=-999, random_state=42, n_jobs=-1
        ))
    ])

    param_grid = {
        'classifier__n_estimators':     [300, 500],
        'classifier__max_depth':        [9, 11],
        'classifier__learning_rate':    [0.15, 0.2],
        'classifier__subsample':        [0.8],
        'classifier__colsample_bytree': [0.8],
    }

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    n_combos = 1
    for v in param_grid.values(): n_combos *= len(v)
    print(f'[{label}] Grid search: {n_combos} combos × 3-fold CV...')

    gs = GridSearchCV(pipeline, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=0)
    gs.fit(X_train, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test)

    print(f'[{label}] Best params : {gs.best_params_}')
    print(f'[{label}] Best CV f1  : {gs.best_score_:.4f}')
    print(f'\n--- {label} Classification Report ---')
    from sklearn.preprocessing import LabelEncoder as LE_
    le_tmp = LE_()
    le_tmp.classes_ = gs.best_estimator_.named_steps['classifier'].classes_
    return best, X_test, y_test, y_pred, gs.best_score_

print('Training pipeline function ready.')


### 7a. Baseline model (no LW features)

In [ ]:
X_base, y_base, le_base, num_base, cat_base = build_feature_matrix(df_predict, exclude_lw=True)
model_base, X_test_base, y_test_base, y_pred_base, cv_base = train_pipeline(
    X_base, y_base, num_base, cat_base, label='BASELINE'
)
print(classification_report(le_base.inverse_transform(y_test_base), le_base.inverse_transform(y_pred_base)))


### 7b. LW-Enhanced model

In [ ]:
X_lw, y_lw, le_lw, num_lw, cat_lw = build_feature_matrix(df_predict, exclude_lw=False)
model_lw, X_test_lw, y_test_lw, y_pred_lw, cv_lw = train_pipeline(
    X_lw, y_lw, num_lw, cat_lw, label='LW-ENHANCED'
)
print(classification_report(le_lw.inverse_transform(y_test_lw), le_lw.inverse_transform(y_pred_lw)))


## 8. Model Comparison

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

f1_base = f1_score(y_test_base, y_pred_base, average='weighted')
f1_lw   = f1_score(y_test_lw,   y_pred_lw,   average='weighted')
acc_base = accuracy_score(y_test_base, y_pred_base)
acc_lw   = accuracy_score(y_test_lw,   y_pred_lw)

print('='*50)
print(f'{"Model":<25} {"Weighted F1":>12} {"Accuracy":>10}')
print('-'*50)
print(f'{"Baseline (no LW)":<25} {f1_base:>12.4f} {acc_base:>10.4f}')
print(f'{"LW-Enhanced":<25} {f1_lw:>12.4f} {acc_lw:>10.4f}')
print('='*50)
print(f'Delta F1 (LW - Baseline): {f1_lw - f1_base:+.4f}')


In [ ]:
# --- Confusion matrices side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = le_base.classes_

for ax, y_t, y_p, le_, title in [
    (axes[0], y_test_base, y_pred_base, le_base, 'Baseline (no LW)'),
    (axes[1], y_test_lw,   y_pred_lw,   le_lw,   'LW-Enhanced'),
]:
    cm = confusion_matrix(y_t, y_p)
    names = le_.inverse_transform([0,1,2])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=names, yticklabels=names)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


## 9. Feature Importance — LW-Enhanced Model

In [ ]:
import re as _re

# Extract feature names after one-hot encoding
preprocessor_step = model_lw.named_steps['preprocessor']
ohe               = preprocessor_step.named_transformers_['cat'].named_steps['onehot']
cat_names         = ohe.get_feature_names_out(cat_lw).tolist()
all_feature_names = num_lw + cat_names

xgb_clf    = model_lw.named_steps['classifier']
importances = xgb_clf.feature_importances_

fi_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)

# Highlight LW features
lw_feature_names = ['LW_Count_On_Date','Total_LW_Mins_On_Date','LW_Day_Had_Warning',
                    'LW_Active_At_Sched_Departure','LW_Overlap_With_Ground_Window_Mins',
                    'LW_Active_During_Ground_Time','Mins_Since_Last_LW_Before_Dep']
fi_df['is_lw'] = fi_df['feature'].apply(lambda f: any(lf in f for lf in lw_feature_names))

# Top-30 chart
top30 = fi_df.head(30)
colors = ['#e74c3c' if lw else '#2980b9' for lw in top30['is_lw']]

plt.figure(figsize=(10, 9))
plt.barh(top30['feature'][::-1], top30['importance'][::-1], color=colors[::-1])
plt.xlabel('XGBoost Feature Importance (gain)')
plt.title('Top 30 Features — LW-Enhanced Model\n(red = Lightning Warning features)')
plt.tight_layout()
plt.show()

# LW-only importance
lw_fi = fi_df[fi_df['is_lw']].copy()
print('\nLightning Warning feature importances:')
print(lw_fi[['feature','importance']].to_string(index=False))


## 10. Save the LW-Enhanced Model

In [ ]:
model_bundle = {
    'pipeline':          model_lw,
    'label_encoder':     le_lw,
    'feature_columns':   X_lw.columns.tolist(),
    'numeric_feats':     num_lw,
    'categorical_feats': cat_lw,
    'lw_feature_names':  lw_feature_names,
    'cv_f1_weighted':    cv_lw,
}

import os
os.makedirs(os.path.dirname(path_model_out), exist_ok=True)
with open(path_model_out, 'wb') as f:
    pickle.dump(model_bundle, f)

print(f'Model saved to: {path_model_out}')
print(f'Model bundle keys: {list(model_bundle.keys())}')


## Done!

The LW-enhanced model bundle has been saved. To use it in the dashboard:
1. Copy `model_with_lw.pkl` to `data/model.pkl` (replacing the existing model), **or**
2. Point `loader.py` to `model_with_lw.pkl` in a separate dashboard entry.

The bundle is fully compatible with the existing `10_Delay_Predictor.py` page — the
`feature_columns`, `numeric_feats`, and `categorical_feats` keys are all present, and
the LW features will be set to `0` (no warning) by default in the predictor form,
which gives a conservative baseline estimate.
